In [112]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
import joblib

In [113]:
CATEGORY_WEIGHTS = {
    "Hatchback": 1200, 
    "Sedan": 1600,     
    "SUV": 2100,       
    "Truck": 2500      
}
#Global Configuration

In [114]:
df = pd.read_csv('EV_Energy_Consumption_Dataset.csv')
df.head()

,Vehicle_ID,Timestamp,Speed_kmh,Acceleration_ms2,Battery_State_%,Battery_Voltage_V,Battery_Temperature_C,Driving_Mode,Road_Type,Traffic_Condition,Slope_%,Weather_Condition,Temperature_C,Humidity_%,Wind_Speed_ms,Tire_Pressure_psi,Vehicle_Weight_kg,Distance_Travelled_km,Energy_Consumption_kWh
0,1102,2024-01-01 00:00:00,111.507366,-2.773816,30.415148,378.091525,25.314786,2,1,1,6.879446,4,0.741770,42.172533,7.829253,31.112020,1822.967368,20.757508,12.054317
1,1435,2024-01-01 00:01:00,48.612323,-0.796982,97.385534,392.718377,18.240755,1,2,1,-3.007212,4,-3.495516,57.018427,4.495572,31.504366,2091.831914,0.642918,4.488701
2,1860,2024-01-01 00:02:00,108.733320,0.253800,84.912600,398.993495,44.449145,1,1,3,0.029585,1,9.248275,69.028911,5.144489,33.838015,1816.702497,40.842824,11.701377
3,1270,2024-01-01 00:03:00,38.579484,-2.111395,28.777904,358.128273,28.980155,1,2,2,8.271943,3,2.868409,86.638349,4.518283,33.256014,1283.102642,5.305229,7.389266
4,1106,2024-01-01 00:04:00,57.172438,1.477883,29.740160,310.888162,33.184551,2,1,1,2.776814,2,16.750244,27.189185,4.263406,33.579678,2160.350788,5.825926,6.761205


In [115]:
df.shape

(5000, 19)

In [116]:
df.duplicated().sum()

np.int64(0)

In [117]:
df.isnull().sum()

Vehicle_ID                0
Timestamp                 0
Speed_kmh                 0
Acceleration_ms2          0
Battery_State_%           0
Battery_Voltage_V         0
Battery_Temperature_C     0
Driving_Mode              0
Road_Type                 0
Traffic_Condition         0
Slope_%                   0
Weather_Condition         0
Temperature_C             0
Humidity_%                0
Wind_Speed_ms             0
Tire_Pressure_psi         0
Vehicle_Weight_kg         0
Distance_Travelled_km     0
Energy_Consumption_kWh    0
dtype: int64

In [118]:
# STEP 2: Data Cleaning & Feature Engineering ---
# We need to predict "Efficiency" (Energy used per km)
# Formula: Efficiency = Total Energy / Distance

In [119]:
# Filter 1: Remove rows with 0 distance to avoid division by zero errors
df = df[df['Distance_Travelled_km'] > 0.1]

In [120]:
# --- STEP 3: Select Features (Inputs) ---
# We now include 'Distance' as an input. 
# The model will learn: High Distance + High Speed = High Energy
features = [
    'Distance_Travelled_km', 
    'Speed_kmh', 
    'Traffic_Condition',
    'Temperature_C',
    'Road_Type',
    'Vehicle_Weight_kg'
]

In [ ]:
X = df[features]
y = df['Energy_Consumption_kWh'] # DIRECT TARGET

In [122]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [123]:
# Random Forest is much better than Linear Regression for complex real data
from sklearn.ensemble import RandomForestRegressor 
print("Training Random Forest Model...")
model = RandomForestRegressor(n_estimators=150, random_state=42)
model.fit(X_train, y_train)

Training Random Forest Model...


RandomForestRegressor(n_estimators=150, random_state=42)

In [124]:
score = model.score(X_test, y_test)
print(f"Model R^2 Score: {score:.4f}")

Model R^2 Score: 0.8200


In [125]:
# --- STEP 7: Save ---
joblib.dump(model, 'universal_ev_model.pkl')
print("Model saved as 'universal_ev_model.pkl'")

Model saved as 'universal_ev_model.pkl'


In [126]:
# --- RE-DEFINE THE FUNCTION WITH THE RETURN STATEMENT ---
def get_predicted_range(category, speed, temp, battery_size_kwh):
    # 1. Get Weight
    weight = CATEGORY_WEIGHTS.get(category, 1800)
    
    # 2. Predict Energy
    inputs = [[100, speed, 2, temp, 1, weight]]
    predicted_energy_100km = model.predict(inputs)[0]
    
    # 3. Calculate Range
    estimated_range = (battery_size_kwh / predicted_energy_100km) * 100
    
    return estimated_range  # <--- MAKE SURE THIS IS HERE!

# --- NOW RUN THE TEST AGAIN ---
print("\n--- FINAL VERIFICATION ---")

# Test 1: Tata Tiago EV (Hatchback)
tata_range = get_predicted_range("Hatchback", speed=80, temp=25, battery_size_kwh=24)
print(f"Tata Tiago EV (Hatchback) Range: {tata_range:.1f} km")

# Test 2: Mahindra XEV 9e (SUV)
mah_range = get_predicted_range("SUV", speed=80, temp=25, battery_size_kwh=79)
print(f"Mahindra XEV 9e (SUV) Range:     {mah_range:.1f} km")


--- FINAL VERIFICATION ---
Tata Tiago EV (Hatchback) Range: 214.3 km
Mahindra XEV 9e (SUV) Range:     709.5 km


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
